# 2022 COE Review AEP Weather Benchmark Working Analysis

If you want to run this notebook, keep in mind that the library path from the FLORIS config file has to be changed accordingly to your new path

## Imports and environment set up

In [1]:
from pathlib import Path
from itertools import product

import pandas as pd
import matplotlib.pyplot as plt

from whale import Project
from whale.utilities import load_yaml

pd.options.display.float_format = '{:,.4f}'.format
pd.options.display.max_columns = 100
pd.options.display.max_rows = 100

%load_ext autoreload
%autoreload 2

# Function to Extract Results

In [2]:
def extract_results(project_name):  # Run the project and clean up the logging
    #project_name = "UMaine_catenary"
    library_path = Path("library").resolve()
    
    metrics_configuration = {
        "# Turbines": {"metric": "n_turbines", "kwargs": {}},
        "Turbine Rating (MW)": {"metric": "turbine_rating", "kwargs": {}},
        "Project Capacity (MW)": {"metric": "capacity", "kwargs": {"units": "mw"}},
        "# OSS": {"metric": "n_substations", "kwargs": {}},
        "Total Export Cable Length (km)": {"metric": "export_system_total_cable_length", "kwargs": {}},
        "Total Array Cable Length (km)": {"metric": "array_system_total_cable_length", "kwargs": {}},
        "CapEx ($)": {"metric": "capex", "kwargs": {}},
        "CapEx per kW ($/kW)": {"metric": "capex", "kwargs": {"per_capacity": "kw"}},
        "OpEx ($)": {"metric": "opex", "kwargs": {}},
        "OpEx per kW ($/kW)": {"metric": "opex", "kwargs": {"per_capacity": "kw"}},
        "AEP (MWh)": {
            "metric": "energy_production",
            "kwargs": {"units": "mw", "aep": True, "with_losses": True}
        },
        "AEP per kW (MWh/kW)": {
            "metric": "energy_production",
            "kwargs": {"units": "mw", "per_capacity": "kw", "aep": True, "with_losses": True}
        },
        "Net Capacity Factor Without Unmodeled Losses (%)": {
            "metric": "capacity_factor",
            "kwargs": {"which": "net"}
        },
        "Net Capacity Factor With All Losses (%)": {
            "metric": "capacity_factor",
            "kwargs": {"which": "net", "with_losses": True}
        },
        "Gross Capacity Factor (%)": {"metric": "capacity_factor", "kwargs": {"which": "gross"}},
        "Energy Availability (%)": {"metric": "availability", "kwargs": {"which": "energy"}},
        "LCOE ($/MWh)": {"metric": "lcoe", "kwargs": {}},
        "IRR (%)": {"metric": "irr", "kwargs": {}},
        "NPV ($)": {"metric": "npv", "kwargs": {}},
    }

    metrics_order = [
        "# Turbines",
        "Turbine Rating (MW)",
        "Project Capacity (MW)",
        "# OSS",
        "Total Export Cable Length (km)",
        "Total Array Cable Length (km)",
        "FCR (%)",
        "Offtake Price ($/MWh)",
        "CapEx ($)",
        "CapEx per kW ($/kW)",
        "System CapEx for Export Cables ($)",
        "Installation CapEx for Export Cables ($)",
        "CapEx Without Export Cables ($)",
        "OpEx ($)",
        "OpEx per kW ($/kW)",
        "Annual OpEx per kW ($/kW)",
        "Energy Availability (%)",
        "Gross Capacity Factor (%)",
        "Net Capacity Factor Without Unmodeled Losses (%)",
        "Net Capacity Factor With All Losses (%)",
        "AEP (MWh)",
        "AEP per kW (MWh/kW)",
        "LCOE ($/MWh)",
        "IRR (%)",
        "NPV ($)",
    ]
    final_cols = ["CapEx ($)", "OpEx ($)", "Energy Production (GWh)", "Revenue ($)", "Cash Flow ($)"]
    
    config = load_yaml(
        library_path / "project/config",
        f"{project_name.replace(' ', '_')}_base.yaml"
    )
    project = Project(
        # Basic Model Configurations
        library_path=library_path,
        #weather_profile=library_path / "weather" / "ocean_wind_1_39.0_-74.0_1959_2023.csv",
        connect_floris_to_layout=True,
        connect_orbit_array_design=True,
        **config,
    )

    
    project.run(
        which_floris="wind_rose",
        full_wind_rose=False,
        floris_reinitialize_kwargs=dict(cut_in_wind_speed=3.0, cut_out_wind_speed=25.0)
    )
    project.wombat.env.cleanup_log_files()  # Delete logging data from WOMBAT
    
    #fig, ax = project.plot_farm(return_fig=True)
    df_opex = project.wombat.metrics.opex(frequency="annual", by_category = True)
    
    display(df_opex)
    
    df_opex.to_csv('library/results/' + project_name + '_OpEx_Breakdown.csv')
    
    # Gather the high-level results
    report_df = project.generate_report(metrics_configuration, project_name).T
    export_system = project.orbit.system_costs["ExportCableInstallation"]
    export_installation = project.orbit.installation_costs["ExportCableInstallation"]
    capex_sans_export = project.orbit.total_capex - export_system - export_installation
    additional_reporting = pd.DataFrame(
        [
            ["FCR (%)", project.fixed_charge_rate],
            ["Offtake Price ($/MWh)", project.offtake_price],
            ["System CapEx for Export Cables ($)", export_system],
            ["Installation CapEx for Export Cables ($)", export_installation],
            ["CapEx Without Export Cables ($)", capex_sans_export],
            ["Annual OpEx per kW ($/kW)", report_df.loc["OpEx per kW ($/kW)", project_name] / project.operations_years],
        ],
        columns=["Project"] + report_df.columns.tolist(),
    ).set_index("Project")
    report_df = pd.concat((report_df, additional_reporting), axis=0).loc[metrics_order]

    # Gather the detailed results
    monthly_results = project.cash_flow(breakdown=True).join(project.energy_production(frequency="month-year")).fillna(0)
    monthly_results = monthly_results.assign(
        CapEx_Installation=monthly_results[[c for c in monthly_results if c.startswith("CapEx") and c.endswith("Installation")]].sum(axis=1),
        CapEx_System=monthly_results[[c for c in monthly_results if c.startswith("CapEx") and c.endswith("System")]].sum(axis=1),
    )

    # monthly_results.to_csv(library_path / "results" / f"{project_name.lower().replace(' ', '_')}_monthly_detailed_results.csv")
    monthly_results["CapEx ($)"] = monthly_results[["CapEx_Installation", "CapEx_Soft", "CapEx_Project", "CapEx_System", "CapEx_Turbine"]].sum(axis=1)
    monthly_results = monthly_results.rename(columns={"OpEx": "OpEx ($)","Revenue": "Revenue ($)", "cash_flow": "Cash Flow ($)"})[final_cols]

    if "ExportSystemDesign" in project.orbit._phases:
        export = "ExportSystemDesign"
    else:
        export = "ElectricalDesign"

    # Create the inputs data
    inputs = pd.DataFrame(
        [
            ["FCR", project.fixed_charge_rate],
            ["Discount rate (%)", project.discount_rate],
            ["# Turbines", project.n_turbines()],
            ["Turbine Rating (MW)", project.turbine_rating()],
            ["Project Capacity (MW)", project.capacity("mw")],
            ["# OSS", project.n_substations()],
            ["Substructure type", "??"],
            ["Row spacing (rotor diameters)", "not used for custom layouts"],
            ["Turbine spacing (rotor diameters)", "not used for custom layouts"],
            ["Depth (m)", project.orbit.config["site"]["depth"]],
            [
                "Mean wind speed (m/s)",
                project.weather.loc[
                    project.orbit_start_date: project.wombat.env.end_datetime,
                    "windspeed_100m"
                ].mean()
            ],
            ["Distance to landfall (km)", project.orbit.config["site"]["distance_to_landfall"]],
            ["Distance to port (km)", project.wombat.config.port_distance],
            ["Interconnection distance (km)", project.orbit._phases[export]._distance_to_interconnection],
            ["# of POIs", "??"],
            ["Export cable type", [*project.orbit._phases[export].cable_lengths_by_type]],
            ["Array cable type", [*project.orbit._phases["CustomArraySystemDesign"].cable_lengths_by_type]],
        ],
        columns=["Inputs", f"{project_name}"]
    ).set_index("Inputs")

    # Save the outputs
    report_df.index.name = "Metrics"
    wrong_indexes = ["Offtake Price ($/MWh)", "System CapEx for Export Cables ($)", "Installation CapEx for Export Cables ($)", "CapEx Without Export Cables ($)", "IRR (%)", "NPV ($)"]
    
    
    return report_df.drop(wrong_indexes,axis=0)

# Extract AEP and OpEx Outputs - Multiple Runs
Availability and OpEx varies depending on the run as it uses probabilistic functions, so we collect the data from 5 runs and calculate the average OpEx and AEP

In [3]:
def display_results_all_runs(n_runs):
    import pandas as pd

    df_list_fixed = list()
    df_final_list = list()
    count = 0
    project = ["COE_2022_fixed_bottom_ERA5_2007_2013", "COE_2022_fixed_bottom_WTK_2007_2013", "COE_2022_fixed_bottom_NOW_23_2007_2013"]
    for i in project:
        df_2 = extract_results(i)
        
        if count == 0:
            df_list_fixed = df_2
        else:
            df_list_fixed = pd.concat([df_list_fixed, df_2], axis=1)
            
        count = count + 1
    
    return df_list_fixed

summary_results = display_results_all_runs(1)


ORBIT library intialized at 'C:\COWER\WHaLE\examples\library'


Missing data in columns ['bury_speed']; all values will be calculated.UserWarning: C:\Users\dmulash\.conda\envs\COWER\lib\site-packages\ORBIT\phases\design\array_system_design.py:906
Missing data in columns ['bury_speed']; all values will be calculated.

Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5


,operations,port_fees,equipment_cost,total_labor_cost,materials_cost,OpEx
year,,,,,,
2007,"3,600,000.0000",0,"25,860,607.2687",0,"735,504.0000","30,196,111.2687"
2008,"3,609,863.0137",0,"16,807,901.3135",0,"480,232.0000","20,897,996.3272"
2009,"3,600,000.0000",0,"68,620,934.0267",0,"1,153,504.0000","73,374,438.0267"
2010,"3,600,000.0000",0,"51,898,513.8572",0,"1,498,600.0000","56,997,113.8572"
2011,"3,600,000.0000",0,"37,975,716.1260",0,"1,052,560.0000","42,628,276.1260"
2012,"3,609,863.0137",0,"57,564,039.5440",0,"1,254,584.0000","62,428,486.5577"
2013,"3,600,000.0000",0,"62,402,275.9098",0,"1,167,312.0000","67,169,587.9098"


Missing data in columns ['bury_speed']; all values will be calculated.UserWarning: C:\Users\dmulash\.conda\envs\COWER\lib\site-packages\ORBIT\phases\design\array_system_design.py:906
Missing data in columns ['bury_speed']; all values will be calculated.

Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5


,operations,port_fees,equipment_cost,total_labor_cost,materials_cost,OpEx
year,,,,,,
2007,"3,600,000.0000",0,"25,860,607.2687",0,"735,504.0000","30,196,111.2687"
2008,"3,609,863.0137",0,"16,807,901.3135",0,"480,232.0000","20,897,996.3272"
2009,"3,600,000.0000",0,"68,620,934.0267",0,"1,153,504.0000","73,374,438.0267"
2010,"3,600,000.0000",0,"51,898,513.8572",0,"1,498,600.0000","56,997,113.8572"
2011,"3,600,000.0000",0,"37,975,716.1260",0,"1,052,560.0000","42,628,276.1260"
2012,"3,609,863.0137",0,"57,564,039.5440",0,"1,254,584.0000","62,428,486.5577"
2013,"3,600,000.0000",0,"62,402,275.9098",0,"1,167,312.0000","67,169,587.9098"


Missing data in columns ['bury_speed']; all values will be calculated.UserWarning: C:\Users\dmulash\.conda\envs\COWER\lib\site-packages\ORBIT\phases\design\array_system_design.py:906
Missing data in columns ['bury_speed']; all values will be calculated.

Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5


,operations,port_fees,equipment_cost,total_labor_cost,materials_cost,OpEx
year,,,,,,
2007,"3,600,000.0000",0,"25,860,607.2687",0,"735,504.0000","30,196,111.2687"
2008,"3,609,863.0137",0,"16,807,901.3135",0,"480,232.0000","20,897,996.3272"
2009,"3,600,000.0000",0,"68,620,934.0267",0,"1,153,504.0000","73,374,438.0267"
2010,"3,600,000.0000",0,"51,898,513.8572",0,"1,498,600.0000","56,997,113.8572"
2011,"3,600,000.0000",0,"37,975,716.1260",0,"1,052,560.0000","42,628,276.1260"
2012,"3,609,863.0137",0,"57,564,039.5440",0,"1,254,584.0000","62,428,486.5577"
2013,"3,600,000.0000",0,"62,402,275.9098",0,"1,167,312.0000","67,169,587.9098"


In [4]:
display(summary_results)

,COE_2022_fixed_bottom_ERA5_2007_2013,COE_2022_fixed_bottom_WTK_2007_2013,COE_2022_fixed_bottom_NOW_23_2007_2013
Metrics,,,
# Turbines,50.0000,50.0000,50.0000
Turbine Rating (MW),12.0000,12.0000,12.0000
Project Capacity (MW),600.0000,600.0000,600.0000
# OSS,1.0000,1.0000,1.0000
Total Export Cable Length (km),118.0680,118.0680,118.0680
Total Array Cable Length (km),277.9831,277.9831,277.9831
FCR (%),0.0569,0.0569,0.0569
CapEx ($),"2,290,237,187.4358","2,290,237,187.4358","2,290,237,187.4358"
CapEx per kW ($/kW),"3,817.0620","3,817.0620","3,817.0620"


In [6]:
summary_results.to_csv('library/results/weather_databases_fixed_bottom_AEP_summary_results.csv')

In [7]:
def display_results_all_runs_all_data(n_runs):
    import pandas as pd

    df_list_fixed = list()
    df_final_list = list()
    count = 0
    project = ["COE_2022_fixed_bottom_ERA5_1959_2022", "COE_2022_fixed_bottom_WTK_2007_2013", "COE_2022_fixed_bottom_NOW_23_2000_2020"]
    for i in project:
        df_2 = extract_results(i)
        
        if count == 0:
            df_list_fixed = df_2
        else:
            df_list_fixed = pd.concat([df_list_fixed, df_2], axis=1)
            
        count = count + 1
    
    return df_list_fixed

summary_results = display_results_all_runs_all_data(1)


Missing data in columns ['bury_speed']; all values will be calculated.UserWarning: C:\Users\dmulash\.conda\envs\COWER\lib\site-packages\ORBIT\phases\design\array_system_design.py:906
Missing data in columns ['bury_speed']; all values will be calculated.

Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5


,operations,port_fees,equipment_cost,total_labor_cost,materials_cost,OpEx
year,,,,,,
1959,"3,600,000.0000",0,"20,862,313.6170",0,"449,600.0000","24,911,913.6170"
1960,"3,609,863.0137",0,"17,780,940.9087",0,"124,000.0000","21,514,803.9224"
1961,"3,600,000.0000",0,"64,943,670.0991",0,"977,800.0000","69,521,470.0991"
1962,"3,600,000.0000",0,"33,258,722.4457",0,"753,200.0000","37,611,922.4457"
1963,"3,600,000.0000",0,"40,146,193.2925",0,"488,200.0000","44,234,393.2925"
1964,"3,609,863.0137",0,"64,866,686.2316",0,"938,800.0000","69,415,349.2453"
1965,"3,600,000.0000",0,"48,411,473.8221",0,"522,600.0000","52,534,073.8221"
1966,"3,600,000.0000",0,"27,730,235.2632",0,"326,000.0000","31,656,235.2632"
1967,"3,600,000.0000",0,"45,359,227.9458",0,"276,600.0000","49,235,827.9458"


Missing data in columns ['bury_speed']; all values will be calculated.UserWarning: C:\Users\dmulash\.conda\envs\COWER\lib\site-packages\ORBIT\phases\design\array_system_design.py:906
Missing data in columns ['bury_speed']; all values will be calculated.

Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5


,operations,port_fees,equipment_cost,total_labor_cost,materials_cost,OpEx
year,,,,,,
2007,"3,600,000.0000",0,"24,779,996.0088",0,"741,504.0000","29,121,500.0088"
2008,"3,609,863.0137",0,"27,983,359.0514",0,"545,752.0000","32,138,974.0651"
2009,"3,600,000.0000",0,"63,680,565.6807",0,"1,135,024.0000","68,415,589.6807"
2010,"3,600,000.0000",0,"50,942,831.4034",0,"1,543,240.0000","56,086,071.4034"
2011,"3,600,000.0000",0,"42,560,506.9646",0,"768,480.0000","46,928,986.9646"
2012,"3,609,863.0137",0,"41,050,734.8964",0,"1,139,064.0000","45,799,661.9101"
2013,"3,600,000.0000",0,"52,414,423.0955",0,"1,105,344.0000","57,119,767.0955"


Missing data in columns ['bury_speed']; all values will be calculated.UserWarning: C:\Users\dmulash\.conda\envs\COWER\lib\site-packages\ORBIT\phases\design\array_system_design.py:906
Missing data in columns ['bury_speed']; all values will be calculated.

Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5


,operations,port_fees,equipment_cost,total_labor_cost,materials_cost,OpEx
year,,,,,,
2000,"3,609,863.0137",0,"28,540,191.9684",0,"741,024.0000","32,891,078.9821"
2001,"3,600,000.0000",0,"19,089,341.0630",0,"500,952.0000","23,190,293.0630"
2002,"3,600,000.0000",0,"60,787,783.3586",0,"1,153,424.0000","65,541,207.3586"
2003,"3,600,000.0000",0,"48,961,915.2294",0,"1,418,680.0000","53,980,595.2294"
2004,"3,609,863.0137",0,"39,802,603.7985",0,"767,360.0000","44,179,826.8122"
2005,"3,600,000.0000",0,"55,018,890.0945",0,"1,016,344.0000","59,635,234.0945"
2006,"3,600,000.0000",0,"59,082,751.5139",0,"1,173,312.0000","63,856,063.5139"
2007,"3,600,000.0000",0,"28,092,964.0969",0,"813,400.0000","32,506,364.0969"
2008,"3,609,863.0137",0,"25,598,564.3930",0,"592,800.0000","29,801,227.4067"


In [8]:
display(summary_results)

,COE_2022_fixed_bottom_ERA5_1959_2022,COE_2022_fixed_bottom_WTK_2007_2013,COE_2022_fixed_bottom_NOW_23_2000_2020
Metrics,,,
# Turbines,50.0000,50.0000,50.0000
Turbine Rating (MW),12.0000,12.0000,12.0000
Project Capacity (MW),600.0000,600.0000,600.0000
# OSS,1.0000,1.0000,1.0000
Total Export Cable Length (km),118.0680,118.0680,118.0680
Total Array Cable Length (km),277.9831,277.9831,277.9831
FCR (%),0.0569,0.0569,0.0569
CapEx ($),"2,290,237,187.4358","2,290,237,187.4358","2,290,237,187.4358"
CapEx per kW ($/kW),"3,817.0620","3,817.0620","3,817.0620"


In [9]:
summary_results.to_csv('library/results/weather_databases_all_years_fixed_bottom_AEP_summary_results.csv')